##### 1: 모자이크데이터 증강시도 

train_transforms = A.Compose([
    A.Resize(640, 640),
    
    # 강력한 기하학적 변형 (p=0.8로 상향하여 모델을 더 강하게 훈련시킵니다)
    A.ShiftScaleRotate(
        shift_limit=0.1, 
        scale_limit=0.2, 
        rotate_limit=30, 
        border_mode=0, # 여백을 검은색으로 채워 알약 강조
        p=0.8
    ),
    
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    
    # 색상 및 디테일 (CLAHE는 알약 각인 식별에 최고입니다)
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
    A.CLAHE(clip_limit=4.0, p=0.4), 

    # 노이즈 (실제 촬영 환경 대비)
    A.GaussNoise(p=0.3),
    A.MotionBlur(blur_limit=3, p=0.2),
    
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
], bbox_params=A.BboxParams(format='coco', label_fields=['category_ids']))

##### 2. 증강코드 해설 

##### 1. 크기 조정 및 위치 변형 (Geometry)
- A.Resize(640, 640): 모든 이미지를 동일한 크기로 맞춥니다. 모델 입력 규격을 맞추는 기초 단계입니다.
- A.ShiftScaleRotate (p=0.8): 가장 강력한 도구입니다. 80% 확률로 실행됩니다.
- Shift: 이미지를 상하좌우로 살짝 밀기 (알약이 정중앙에 없을 때 대비)
- Scale: 크기를 20% 내외로 키우거나 줄이기 (줌인/줌아웃 효과)
- Rotate: 최대 30도 회전 (기우뚱하게 찍힌 경우 대비)
- border_mode=0: 회전 시 생기는 빈 공간을 검은색으로 채웁니다. 알약 배경이 보통 어둡기 때문에 모델이 배경과 물체를 분리하는 데 도움을 줍니다.

##### 2. 반전 및 90도 회전 (Flips & Rotate)
- 알약은 위아래 구분이 없는 경우가 많아 데이터 양을 늘리는 데 최고입니다.
- A.Horizontal/VerticalFlip: 좌우/상하 반전입니다.
- A.RandomRotate90: 90도, 180도, 270도 무작위 회전입니다. 알약이 가로로 놓였든 세로로 놓였든 똑같이 인식하게 합니다.

##### 3. 색상 및 질감 개선 (Color & Detail)
- 알약의 색상 차이와 표면의 각인(글자, 로고)을 선명하게 학습시킵니다.
- A.ColorJitter: 밝기, 대비, 채도를 무작위로 바꿉니다. (노란 조명, 형광등 조명 등 환경 변화 대응)
- A.CLAHE (중요!): 이미지의 대비를 국부적으로 높여줍니다. 알약 표면의 흐릿한 글자나 각인을 선명하게 만들어주는 핵심 기능입니다.

##### 4. 실제 환경 노이즈 (Noise & Blur)
- 화질이 안 좋은 카메라나 흔들린 사진에 대비합니다.
- A.GaussNoise: 디지털 노이즈(지지직거림)를 추가합니다.
- A.MotionBlur: 셔터를 누를 때 손이 흔들려 번진 효과를 줍니다.


##### 3. 학습결과

- [Epoch 1/30] train_loss: 0.7427 | val_loss: 0.4114
- [Epoch 2/30] train_loss: 0.2968 | val_loss: 0.2188
- [Epoch 3/30] train_loss: 0.5586 | val_loss: 0.2499
- [Epoch 4/30] train_loss: 0.3322 | val_loss: 0.1166
- [Epoch 5/30] train_loss: 0.3084 | val_loss: 0.0913
- [Epoch 6/30] train_loss: 0.3959 | val_loss: 0.1038
- [Epoch 7/30] train_loss: 0.2957 | val_loss: 0.0945
- [Epoch 8/30] train_loss: 0.2052 | val_loss: 0.0895
- [Epoch 9/30] train_loss: 0.5605 | val_loss: 0.0809
- [Epoch 10/30] train_loss: 0.3818 | val_loss: 0.0813
- [Epoch 11/30] train_loss: 0.3143 | val_loss: 0.0808
- [Epoch 12/30] train_loss: 0.3877 | val_loss: 0.0803
- [Epoch 13/30] train_loss: 0.5251 | val_loss: 0.0798
- [Epoch 14/30] train_loss: 0.6896 | val_loss: 0.0807
- [Epoch 15/30] train_loss: 0.2956 | val_loss: 0.0776
- [Epoch 16/30] train_loss: 0.4802 | val_loss: 0.0809
- [Epoch 17/30] train_loss: 0.3971 | val_loss: 0.0817
- [Epoch 18/30] train_loss: 0.2980 | val_loss: 0.0817
- [Epoch 19/30] train_loss: 0.1882 | val_loss: 0.0791
- [Epoch 20/30] train_loss: 0.1900 | val_loss: 0.0811
- [Epoch 21/30] train_loss: 0.0897 | val_loss: 0.0813
- [Epoch 22/30] train_loss: 0.6100 | val_loss: 0.0806
- [Epoch 23/30] train_loss: 0.1763 | val_loss: 0.0796
- [Epoch 24/30] train_loss: 0.1139 | val_loss: 0.0802
- [Epoch 25/30] train_loss: 0.8599 | val_loss: 0.0797
- [Epoch 26/30] train_loss: 0.2402 | val_loss: 0.0798
- [Epoch 27/30] train_loss: 0.4055 | val_loss: 0.0808
- [Epoch 28/30] train_loss: 0.3916 | val_loss: 0.0802
- [Epoch 29/30] train_loss: 0.3864 | val_loss: 0.0797
- [Epoch 30/30] train_loss: 0.6269 | val_loss: 0.0804

##### 4 : 오류 발생

- 학습결과는 좋았으나 csv파일 결과가 좋지 않거나 

- map도출결과가 좋지 않음

- 원인을 찾지 못함.

##### 5 : 베이스라인으로 돌아와 yolo모델 재학습 및 모자이크 증강 학습

** 4: YOLO 시리즈 비교 실험 **

아직csv와 map 산출 실패